# Imports

In [15]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/Users/mac/Documents/dev/ID2221/dic/Week 3


In [16]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

26/09/21 15:18:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips_part', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Load dataset file

In [17]:
# load data/yellow_tripdata_1.parquet
taxi_df = spark.read.parquet("../data/yellow_tripdata_1.parquet")
taxi_df.show(5)
taxi_df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

# Summary of dataset

Note that negative values do occur in original dataset so synthetic also having some is to be expected

In [18]:
# summary of dataset
taxi_df.describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|          VendorID|   passenger_count|     trip_distance|       RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+----

# Generate rows

Decimal values are handled with columns values from normal distribution (mean, std) delimited by observed real range  

String values are handled with categorical distribution

Integer values are handled with categorical distribution

Timestamps are handled as any date between max and min with dropoff always equal to pickup + average_trip_duration 

In [ ]:
import random
import string
from datetime import timedelta
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, TimestampType, TimestampNTZType, FloatType, ShortType, DecimalType

ten_percent = int(0.10 * taxi_df.count())


existing_values_and_stats = {}

# find the existing values and stats for each column in taxi_df
for field in taxi_df.schema.fields:
    if isinstance(field.dataType, StringType):
        existing_values_and_stats[field.name] = {
            "Values": taxi_df.select(field.name).distinct().rdd.flatMap(lambda x: x).collect()
        }

    elif isinstance(field.dataType, (IntegerType, LongType, ShortType)):
        existing_values = taxi_df.select(field.name).distinct().rdd.flatMap(lambda x: x).collect()
        existing_values_and_stats[field.name] = {
            "Values": existing_values
        }
    elif isinstance(field.dataType, (DoubleType, FloatType, DecimalType)):
        # calculate mean and std of the column
        mean_std = taxi_df.select(F.mean(field.name), F.stddev(field.name)).first()
        mean = mean_std[0] if mean_std else None
        stddev = mean_std[1] if mean_std else None
        if mean is None or stddev is None:
            # error
            raise ValueError(f"Cannot calculate mean and stddev for column {field.name}")
        else:
            existing_values_and_stats[field.name] = {
                "Mean": mean,
                "Stddev": stddev,
                "max": taxi_df.select(F.max(field.name)).first()[0],
                "min": taxi_df.select(F.min(field.name)).first()[0]
            }
    elif isinstance(field.dataType, (TimestampType, TimestampNTZType)): 
        # pick random timestamp value from range of the existing values in the column
        min_value = taxi_df.select(F.min(field.name)).first()[0]
        max_value = taxi_df.select(F.max(field.name)).first()[0]
        if min_value is None or max_value is None:
            # error
            raise ValueError(f"Cannot calculate min and max for column {field.name}")
        else:
            existing_values_and_stats[field.name] = {
                "min": min_value,
                "max": max_value
            }
    else:
        # error for unsupported data types
        raise ValueError(f"Unsupported data type: {field.dataType}")
    
print("Computed existing values and stats for each column in taxi_df")
    

from pyspark.sql import functions as F


def uniform_from_values(values, data_type, seed=None):
    n = len(values)

    return F.element_at(
        F.array(*[F.lit(v).cast(data_type) for v in values]),
        (F.rand(seed) * n).cast("int") + 1,
    )

def normal_from_stats(existing_values_and_stats, column_name, seed=None):
    mean = existing_values_and_stats[column_name]["Mean"]
    stddev = existing_values_and_stats[column_name]["Stddev"]
    min_value = existing_values_and_stats[column_name]["min"]
    max_value = existing_values_and_stats[column_name]["max"]

    return F.least(
        F.lit(max_value),
        F.greatest(
            F.lit(min_value),
            F.lit(mean) + F.lit(stddev) * F.randn(seed)
        )
    )

average_trip_duration = (taxi_df.select(
        F.mean(F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))
    ).first()[0].__round__())


seed = 42
generated_df = (
    spark.range(ten_percent)
    # Timestamp columns: pick random date from the range defined by max and min in the column
    .withColumn(
    "tpep_pickup_datetime",
    F.to_timestamp(
        F.from_unixtime(
            F.lit(existing_values_and_stats["tpep_pickup_datetime"]["min"].timestamp())
            + F.rand(seed)
            * (
                existing_values_and_stats["tpep_pickup_datetime"]["max"].timestamp()
                - existing_values_and_stats["tpep_pickup_datetime"]["min"].timestamp()
            )
        )
    ).cast("timestamp_ntz")
    )
    .withColumn(
        "tpep_dropoff_datetime",
        F.col("tpep_pickup_datetime")
        + F.expr(f"INTERVAL {average_trip_duration} SECONDS")
    )
    # Categorical columns: pick random value from the existing values in the column
    .withColumn(
        "VendorID",
        uniform_from_values(existing_values_and_stats["VendorID"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "passenger_count",
        uniform_from_values(existing_values_and_stats["passenger_count"]["Values"], LongType(), seed)
    )
    .withColumn(
        "RatecodeID",
        uniform_from_values(existing_values_and_stats["RatecodeID"]["Values"], LongType(), seed)
    )
    .withColumn(
        "store_and_fwd_flag",
        uniform_from_values(existing_values_and_stats["store_and_fwd_flag"]["Values"], StringType(), seed)
    )
    .withColumn(
        "PULocationID",
        uniform_from_values(existing_values_and_stats["PULocationID"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "DOLocationID",
        uniform_from_values(existing_values_and_stats["DOLocationID"]["Values"], IntegerType(), seed)
    )
    .withColumn(
        "payment_type",
        uniform_from_values(existing_values_and_stats["payment_type"]["Values"], LongType(), seed)
    )
    # Numerical columns: generate random value from normal distribution with mean and stddev delimited by the observed min and max values of the column
    .withColumn(
        "trip_distance",
        normal_from_stats(existing_values_and_stats, "trip_distance", seed)
    )
    .withColumn(
        "fare_amount",
        normal_from_stats(existing_values_and_stats, "fare_amount", seed)
    ).withColumn(
        "extra",
        normal_from_stats(existing_values_and_stats, "extra", seed)
    ).withColumn(
        "mta_tax",
        normal_from_stats(existing_values_and_stats, "mta_tax", seed)
    ).withColumn(
        "tip_amount",
        normal_from_stats(existing_values_and_stats, "tip_amount", seed)
    ).withColumn(
        "tolls_amount",
        normal_from_stats(existing_values_and_stats, "tolls_amount", seed)
    ).withColumn(
        "improvement_surcharge",
        normal_from_stats(existing_values_and_stats, "improvement_surcharge", seed)
    ).withColumn(
        "total_amount",
        normal_from_stats(existing_values_and_stats, "total_amount", seed)
    ).withColumn(
        "congestion_surcharge",
        normal_from_stats(existing_values_and_stats, "congestion_surcharge", seed)
    ).withColumn(
        "Airport_fee",
        normal_from_stats(existing_values_and_stats, "Airport_fee", seed)
    )
)

generated_df.show(10)
generated_df.printSchema()

Computed existing values and stats for each column in taxi_df
+---+--------------------+---------------------+--------+---------------+----------+------------------+------------+------------+------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+--------------------+
| id|tpep_pickup_datetime|tpep_dropoff_datetime|VendorID|passenger_count|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|     trip_distance|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|         Airport_fee|
+---+--------------------+---------------------+--------+---------------+----------+------------------+------------+------------+------------+------------------+------------------+------------------+-------------------+------------------+----------------

# Insert duplicates from original dataset

# Write dataframe to parquet file